In [3]:
import pandas as pd
import numpy as np
from faker import Faker

fake = Faker()
np.random.seed(42)

# --------------------------
# Configuración
# --------------------------

NUM_RESTAURANTS = 20
NUM_DAYS = 10
MAX_SALES_PER_DAY = 20

# --------------------------
# Restaurantes
# --------------------------

restaurants = []

for i in range(1, NUM_RESTAURANTS + 1):
    restaurants.append({
        "restaurant_id": f"R{i:03d}",
        "restaurant_name": f"Restaurant_{i}",
        "zone": np.random.choice(
            ["Centro","Norte","Sur","Este","Oeste"]
        )
    })

restaurants_df = pd.DataFrame(restaurants)

# --------------------------
# Estados
# --------------------------

sale_statuses = pd.DataFrame({
    "sale_status_id":[1,2,3,4,5,6,7],
    "sale_status":[
        "COMPLETED",
        "CANCELLED",
        "DUPLICATE",
        "REFUNDED",
        "PARTIAL_REFUND",
        "PAYMENT_REJECTED",
        "UNDER_REVIEW"
    ],
    "is_valid_sale":[1,0,0,0,1,0,0]
})

# --------------------------
# Ventas
# --------------------------

records = []

sale_counter = 1

dates = pd.date_range(
    start="2026-01-01",
    periods=NUM_DAYS,
    freq="D"
)

status_probs = [
    0.82,  # completed
    0.05,  # cancelled
    0.03,  # duplicate
    0.03,  # refunded
    0.03,  # partial refund
    0.02,  # rejected
    0.02   # review
]

for day in dates:

    for restaurant in restaurants_df["restaurant_id"]:

        sales_count = np.random.randint(
            1,
            MAX_SALES_PER_DAY + 1
        )

        for _ in range(sales_count):

            status_id = np.random.choice(
                [1,2,3,4,5,6,7],
                p=status_probs
            )

            gross = round(
                np.random.uniform(8,50),
                2
            )

            discount = round(
                gross * np.random.uniform(0,0.20),
                2
            )

            net = gross - discount

            timestamp = fake.date_time_between(
                start_date=day,
                end_date=day + pd.Timedelta(days=1)
            )

            records.append({

                "sale_id":
                    f"S{sale_counter:08d}",

                "restaurant_id":
                    restaurant,

                "sale_timestamp":
                    timestamp,

                "sale_status_id":
                    status_id,

                "customer_count":
                    np.random.randint(1,6),

                "gross_amount":
                    gross,

                "discount_amount":
                    discount,

                "net_amount":
                    round(net,2),

                "payment_method":
                    np.random.choice([
                        "CASH",
                        "CARD",
                        "TRANSFER",
                        "WALLET"
                    ]),

                "channel":
                    np.random.choice([
                        "DINE_IN",
                        "DELIVERY",
                        "PICKUP"
                    ]),

                "source_system":
                    np.random.choice([
                        "POS_A",
                        "POS_B",
                        "APP"
                    ])

            })

            sale_counter += 1

sales_df = pd.DataFrame(records)

# --------------------------
# Crear duplicados reales
# --------------------------

dup_count = int(
    len(sales_df) * 0.03
)

duplicates = sales_df.sample(
    dup_count,
    random_state=42
).copy()

duplicates["sale_status_id"] = 3

sales_df = pd.concat(
    [sales_df, duplicates],
    ignore_index=True
)

# --------------------------
# Ventas canceladas sin monto
# --------------------------

cancel_mask = (
    sales_df["sale_status_id"] == 2
)

sales_df.loc[
    cancel_mask,
    ["gross_amount",
     "discount_amount",
     "net_amount"]
] = 0

# --------------------------
# Reembolsos
# --------------------------

refund_mask = (
    sales_df["sale_status_id"] == 4
)

sales_df.loc[
    refund_mask,
    "net_amount"
] *= -1

# --------------------------
# Casos sucios intencionales
# --------------------------

dirty_rows = sales_df.sample(
    int(len(sales_df)*0.01),
    random_state=42
).index

sales_df.loc[
    dirty_rows,
    "payment_method"
] = None

# --------------------------
# Exportar
# --------------------------

restaurants_df.to_csv(
    "dim_restaurants.csv",
    index=False
)

sale_statuses.to_csv(
    "dim_sale_status.csv",
    index=False
)

sales_df.to_csv(
    "fact_sales.csv",
    index=False
)

print("Archivos generados.")
print(
    restaurants_df.shape,
    sale_statuses.shape,
    sales_df.shape
)

Archivos generados.
(20, 3) (7, 3) (2237, 11)
